# Week 4 Homework - SDXL Pipeline, ControlNet & Evaluation
**COM SCI-X 455 - Week 4**  |  GPU: T4 (Runtime > Change runtime type > T4 GPU)

Theme (linked to my capstone, child care for multilingual families): **home daycare interiors, children's products,
and outdoor play spaces.** No images of children are generated.

All images are saved under `images/` with descriptive names; tables under `images/tables/`.
Optional key in Colab Secrets: `UCLA_OPENAI_API_KEY` (Part 3 Config C uses the OpenAI image API; without a key
it falls back to SDXL Turbo at 4 steps, as the assignment allows).

In [ ]:
!pip install -q -U diffusers openai
!pip install -q lpips tabulate

In [ ]:
import os
def get_secret(*names):
    """Read a key from Colab Secrets (or env vars). Keys are never printed."""
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

OPENAI_KEY = get_secret("UCLA_OPENAI_API_KEY", "OPENAI_API_KEY")
ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY", "UCLA_ANTHROPIC_API_KEY", "CLAUDE_API_KEY")
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
if ANTHROPIC_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY
print("OpenAI key found:", bool(OPENAI_KEY), "| Anthropic key found:", bool(ANTHROPIC_KEY))

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch, time, json, base64, io, gc
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from PIL import Image
from diffusers import (StableDiffusionXLPipeline, StableDiffusionXLControlNetPipeline, ControlNetModel,
                       AutoencoderKL, DPMSolverMultistepScheduler, EulerDiscreteScheduler, AutoPipelineForText2Image)
assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU"
DEVICE = "cuda"
for d in ["images/part1", "images/part2/products", "images/part2/gallery", "images/part2/sweep", "images/part3", "images/tables"]:
    os.makedirs(d, exist_ok=True)

def grid(images, titles, path, cols=None, size=3.6):
    cols = cols or len(images); rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows + 0.3))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im); ax.set_title(t, fontsize=9)
    plt.tight_layout(); fig.savefig(path, dpi=120, bbox_inches="tight"); plt.show()

def save_table(df, name):
    df.to_csv(f"images/tables/{name}.csv", index=False)
    open(f"images/tables/{name}.md", "w").write(df.to_markdown(index=False))

In [ ]:
# If this cell is re-run, free the old copy first (two SDXL copies do not fit on a T4)
for _name in ["pipe", "pipe_cn", "turbo"]:
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", vae=vae,
        torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to(DEVICE)
pipe.set_progress_bar_config(disable=True)
try:
    pipe.vae.enable_slicing()          # newer diffusers: slicing lives on the VAE
except AttributeError:
    pass
SCHEDULERS = {
    "DPM++ 2M": lambda cfg: DPMSolverMultistepScheduler.from_config(cfg, algorithm_type="dpmsolver++", solver_order=2),
    "Euler": lambda cfg: EulerDiscreteScheduler.from_config(cfg),
}
BASE_SCHED_CONFIG = pipe.scheduler.config

def sdxl(prompt, scheduler="DPM++ 2M", seed=42, steps=25, cfg=7.0, negative="blurry, low quality, distorted, text, watermark, people"):
    pipe.scheduler = SCHEDULERS[scheduler](BASE_SCHED_CONFIG)
    g = torch.Generator(DEVICE).manual_seed(seed)
    return pipe(prompt, negative_prompt=negative, num_inference_steps=steps, guidance_scale=cfg, generator=g).images[0]

# CLIP score with openai/clip-vit-base-patch32 (Hugging Face), reported as 100 x cosine similarity
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_score(img, text):
    inp = clip_proc(text=[text], images=img, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    out = clip(**inp)
    i = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    t = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return round(100 * (i @ t.T).item(), 2)
print("SDXL + CLIP ready")

## Part 1 - Multi-Domain Text-to-Image with SDXL

Per domain: 3 prompt formulations (abstract -> specific -> detailed-technical) with DPM++ 2M, plus the specific and
technical prompts again with Euler = **5 images per domain, 15 total**. Seed 42 for every image, 25 steps, CFG 7.

CLIP is measured twice: against the image's own prompt, and against one shared **intent caption** per domain,
so the three prompt formulations can be compared on the same text.

In [ ]:
DOMAINS = {
 "interior": {
   "intent": "a bright home daycare playroom with wooden shelves, play mats and books",
   "abstract": "a cozy place where little kids play",
   "specific": "a bright home daycare playroom with low wooden shelves, soft play mats and picture books, afternoon light",
   "technical": "interior architecture photograph of a home daycare playroom, low natural-wood Montessori shelves, cream play mats, reading corner with floor cushions and picture books, large window with soft afternoon light, wide-angle 24mm lens, f/8, eye level, clean composition, high detail"},
 "product": {
   "intent": "a wooden rainbow stacking toy product photo",
   "abstract": "a fun toy for babies",
   "specific": "a wooden rainbow stacking ring toy on a white table, product photo",
   "technical": "e-commerce product photograph of a wooden rainbow stacking ring toy with 6 pastel rings on a round base, pure white seamless background, softbox lighting from the left, 85mm lens, f/11, sharp focus, centered, soft contact shadow"},
 "outdoor": {
   "intent": "an outdoor daycare play yard with a sandbox and garden",
   "abstract": "a happy outdoor place for children",
   "specific": "an outdoor daycare play yard with a wooden sandbox, small garden beds and a shade sail, sunny morning",
   "technical": "landscape photograph of a backyard daycare play yard, cedar sandbox, raised vegetable garden beds, triangular white shade sail, rubber safety surface, green lawn, morning sun at 30 degrees, 35mm lens, f/8, deep depth of field, natural colors"},
}
p1_rows, p1_imgs = [], {}
for dom, P in DOMAINS.items():
    runs = [("abstract", "DPM++ 2M"), ("specific", "DPM++ 2M"), ("technical", "DPM++ 2M"),
            ("specific", "Euler"), ("technical", "Euler")]
    for level, sch in runs:
        t0 = time.time(); img = sdxl(P[level], scheduler=sch); dt = time.time() - t0
        name = f"{dom}_{level}_{sch.replace(' ', '').replace('+', 'p')}_seed42.png"
        img.save(f"images/part1/{name}"); p1_imgs[(dom, level, sch)] = img
        p1_rows.append({"Domain": dom, "Prompt level": level, "Scheduler": sch, "Seconds": round(dt, 1),
                        "CLIP (own prompt)": clip_score(img, P[level]), "CLIP (intent)": clip_score(img, P["intent"]), "File": name})
    grid([p1_imgs[(dom, l, s)] for l, s in runs], [f"{l} / {s}" for l, s in runs], f"images/part1/{dom}_grid.png")
p1 = pd.DataFrame(p1_rows); save_table(p1, "part1_clip")
p1

In [ ]:
# Prompt comparison (DPM++ 2M only) and scheduler comparison (same prompt + seed)
prompt_cmp = p1[p1.Scheduler == "DPM++ 2M"].pivot(index="Domain", columns="Prompt level", values="CLIP (intent)")[["abstract", "specific", "technical"]]
prompt_cmp["best (intent CLIP)"] = prompt_cmp.idxmax(axis=1)
print("Prompt formulation, CLIP vs intent caption:\n", prompt_cmp.to_markdown())
sch = p1[p1["Prompt level"] != "abstract"].pivot_table(index=["Domain", "Prompt level"], columns="Scheduler", values="CLIP (own prompt)")
sch["Euler - DPM++"] = (sch["Euler"] - sch["DPM++ 2M"]).round(2)
print("\nScheduler comparison, CLIP vs own prompt:\n", sch.to_markdown())
print("\nMean |difference|:", round(sch["Euler - DPM++"].abs().mean(), 2), "CLIP points")
# Pixel-level difference between schedulers (same seed): mean absolute difference 0-255
for dom in DOMAINS:
    a = np.asarray(p1_imgs[(dom, "technical", "DPM++ 2M")], dtype=float); b = np.asarray(p1_imgs[(dom, "technical", "Euler")], dtype=float)
    print(f"{dom}: mean pixel difference DPM++ vs Euler = {np.abs(a - b).mean():.1f} / 255")
save_table(prompt_cmp.reset_index(), "part1_prompt_comparison"); save_table(sch.reset_index(), "part1_scheduler_comparison")

## Part 2 - ControlNet Application: Use case A (E-commerce)

Five children's products, each placed in 3 lifestyle contexts (kitchen, outdoor, studio) with Canny ControlNet = 15
images. **Product photos:** put your own phone photos in `products/` (`product1.jpg` ... `product5.jpg`) to use them.
If that folder is empty, the notebook creates catalog-style reference shots on a white background with SDXL and
uses those as the "store photos" (this is what my committed run used).

In [ ]:
# T4 memory: SDXL + ControlNet do not both fit on the GPU, so move SDXL to CPU and let diffusers
# move one sub-model at a time to the GPU (model CPU offload). The ControlNet pipeline shares SDXL's weights.
pipe.to("cpu"); gc.collect(); torch.cuda.empty_cache()
controlnet = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16, variant="fp16")
pipe_cn = StableDiffusionXLControlNetPipeline.from_pipe(pipe, controlnet=controlnet)
pipe_cn.enable_model_cpu_offload()
pipe_cn.set_progress_bar_config(disable=True)
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# From here on, plain SDXL images use the same pipeline with ControlNet scale 0.0:
# the ControlNet residuals are multiplied by 0, so the output is the base SDXL model.
_blank = Image.new("RGB", (1024, 1024))
def sdxl(prompt, scheduler="DPM++ 2M", seed=42, steps=25, cfg=7.0, negative="blurry, low quality, distorted, text, watermark, people"):
    pipe_cn.scheduler = SCHEDULERS[scheduler](BASE_SCHED_CONFIG)
    g = torch.Generator(DEVICE).manual_seed(seed)
    return pipe_cn(prompt, negative_prompt=negative, image=_blank, controlnet_conditioning_scale=0.0,
                   num_inference_steps=steps, guidance_scale=cfg, generator=g).images[0]

def canny(img, low=100, high=200):
    e = cv2.Canny(np.array(img.convert("RGB")), low, high)
    return Image.fromarray(np.stack([e] * 3, -1))

def cn_generate(prompt, control, scale, seed=42, steps=25, cfg=7.0):
    pipe_cn.scheduler = SCHEDULERS["DPM++ 2M"](BASE_SCHED_CONFIG)
    g = torch.Generator(DEVICE).manual_seed(seed)
    return pipe_cn(prompt, negative_prompt="blurry, low quality, distorted, text, watermark, people, hands",
                   image=control, controlnet_conditioning_scale=scale, num_inference_steps=steps,
                   guidance_scale=cfg, generator=g).images[0]

def edge_f1(img, ref_edges):
    """Structure fidelity: F1 overlap between Canny edges of the output and the reference (3px tolerance)."""
    a = np.array(canny(img))[..., 0] > 0; b = np.array(ref_edges)[..., 0] > 0
    k = np.ones((7, 7), np.uint8)
    prec = (a & cv2.dilate(b.astype(np.uint8), k).astype(bool)).sum() / max(a.sum(), 1)
    rec = (b & cv2.dilate(a.astype(np.uint8), k).astype(bool)).sum() / max(b.sum(), 1)
    return round(2 * prec * rec / max(prec + rec, 1e-9), 3)

PRODUCTS = {
 "stacking_toy": "a wooden rainbow stacking ring toy",
 "sippy_cup": "a mint green toddler sippy cup with two handles",
 "backpack": "a small yellow children's backpack with a fox face",
 "toy_train": "a red and blue wooden toy train with three cars",
 "toddler_chair": "a small natural wood toddler chair",
}
os.makedirs("products", exist_ok=True)
user_photos = sorted(f for f in os.listdir("products") if f.lower().endswith((".jpg", ".jpeg", ".png")))
refs = {}
for i, (key, desc) in enumerate(PRODUCTS.items()):
    if i < len(user_photos):
        img = Image.open(f"products/{user_photos[i]}").convert("RGB").resize((1024, 1024))
    else:
        img = sdxl(f"catalog product photo of {desc}, isolated on pure white background, centered, studio lighting, sharp focus", seed=7)
    img.save(f"images/part2/products/{key}_reference.png"); refs[key] = img
grid([refs[k] for k in PRODUCTS] + [canny(refs[k]) for k in PRODUCTS], list(PRODUCTS) + [k + " edges" for k in PRODUCTS],
     "images/part2/products_and_edges.png", cols=5, size=3)

In [ ]:
CONTEXTS = {
 "kitchen": "on a sunny kitchen counter next to a fruit bowl, warm home lifestyle photo",
 "outdoor": "on a picnic blanket in a green park, soft natural daylight, lifestyle photo",
 "studio": "on a pastel colored studio backdrop with soft shadows, professional product photo",
}
# Conditioning scale sweep on one example: stacking toy in the kitchen
SWEEP = [0.3, 0.5, 0.8, 1.0]
ex_prompt = f"{PRODUCTS['stacking_toy']} {CONTEXTS['kitchen']}"
ex_edges = canny(refs["stacking_toy"])
sweep_rows, sweep_imgs = [], []
for s in SWEEP:
    img = cn_generate(ex_prompt, ex_edges, s)
    img.save(f"images/part2/sweep/stacking_toy_kitchen_scale{s}.png"); sweep_imgs.append(img)
    sweep_rows.append({"Scale": s, "Edge F1 (structure)": edge_f1(img, ex_edges), "CLIP (scene prompt)": clip_score(img, ex_prompt)})
sweep_df = pd.DataFrame(sweep_rows); save_table(sweep_df, "part2_scale_sweep")
grid(sweep_imgs, [f"scale {r['Scale']}: edgeF1 {r['Edge F1 (structure)']}, CLIP {r['CLIP (scene prompt)']}" for r in sweep_rows],
     "images/part2/scale_sweep_grid.png")
sweep_df

In [ ]:
DEPLOY_SCALE = 0.8   # chosen from the sweep: keeps product shape (edge F1) while the scene still changes (see memo)
gal_rows, gal_imgs, gal_titles = [], [], []
for key, desc in PRODUCTS.items():
    edges = canny(refs[key])
    for ctx, ctx_text in CONTEXTS.items():
        prompt = f"{desc} {ctx_text}"
        img = cn_generate(prompt, edges, DEPLOY_SCALE)
        img.save(f"images/part2/gallery/{key}_{ctx}.png")
        gal_imgs.append(img); gal_titles.append(f"{key} / {ctx}")
        gal_rows.append({"Product": key, "Context": ctx, "Edge F1": edge_f1(img, edges), "CLIP": clip_score(img, prompt)})
gal_df = pd.DataFrame(gal_rows); save_table(gal_df, "part2_gallery_scores")
grid(gal_imgs, gal_titles, "images/part2/gallery_grid.png", cols=3, size=3.2)
gal_df.groupby("Context")[["Edge F1", "CLIP"]].mean().round(3)

## Part 3 - Comparative Evaluation Report

5 prompts x 3 configurations = 15 images.
- **A:** SDXL base, guidance 7.5, 30 steps, DPM++ 2M (seed 42)
- **B:** SDXL base + ControlNet Canny (scale 0.7), same prompts. Control image = Canny edges of a simple layout
  sketch drawn with OpenCV for each prompt (so B gets layout help that A does not).
- **C:** OpenAI image API (DALL-E 3, or `gpt-image-1` if DALL-E 3 is retired); SDXL Turbo at 4 steps if no key.

Metrics: CLIP (openai/clip-vit-base-patch32), LPIPS (AlexNet) against the highest-CLIP image for that prompt,
and a 1-5 preference rating (GPT-4o vision pass with a fixed rubric, which I then review image by image).

In [ ]:
P3_PROMPTS = [
 "a storefront sign of a small daycare that reads OPEN SEATS in bold letters",
 "three red apples and two green pears on a wooden kitchen table",
 "a blue cup to the left of a yellow bowl on a white table",
 "a cozy reading nook with a beanbag and a bookshelf, watercolor illustration",
 "a modern single-story house with a fenced backyard playground, golden hour photograph",
]
def layout_sketch(i):
    """Simple white-on-black layout drawings that give Config B a composition to follow."""
    c = np.zeros((1024, 1024, 3), np.uint8); w = (255, 255, 255)
    if i == 0: cv2.rectangle(c, (150, 300), (874, 520), w, 6); cv2.rectangle(c, (250, 600), (774, 1000), w, 6)
    if i == 1:
        for x in (250, 450, 650): cv2.circle(c, (x, 560), 90, w, 6)
        for x in (380, 580): cv2.ellipse(c, (x, 720), (75, 110), 0, 0, 360, w, 6)
        cv2.line(c, (0, 850), (1024, 850), w, 6)
    if i == 2: cv2.rectangle(c, (180, 420), (420, 720), w, 6); cv2.ellipse(c, (690, 640), (200, 90), 0, 0, 360, w, 6); cv2.line(c, (0, 780), (1024, 780), w, 6)
    if i == 3: cv2.rectangle(c, (600, 150), (950, 900), w, 6); cv2.ellipse(c, (330, 760), (220, 140), 0, 0, 360, w, 6); cv2.rectangle(c, (60, 120), (420, 420), w, 6)
    if i == 4:
        pts = np.array([[200, 520], [512, 300], [824, 520]]); cv2.polylines(c, [pts], False, w, 6)
        cv2.rectangle(c, (200, 520), (824, 760), w, 6); cv2.line(c, (0, 880), (1024, 880), w, 6)
        for x in range(0, 1024, 60): cv2.line(c, (x, 820), (x, 900), w, 4)
    return Image.fromarray(c)

oa_client = None
if OPENAI_KEY:
    from openai import OpenAI; oa_client = OpenAI()
turbo = None
def config_c(prompt):
    global turbo
    if oa_client:
        for model in ["dall-e-3", "gpt-image-1"]:
            try:
                kw = dict(model=model, prompt=prompt, size="1024x1024", n=1)
                if model == "dall-e-3": kw["response_format"] = "b64_json"
                r = oa_client.images.generate(**kw)
                return Image.open(io.BytesIO(base64.b64decode(r.data[0].b64_json))).convert("RGB"), model
            except Exception as e:
                print(f"  {model} failed: {str(e)[:120]}")
    if turbo is None:
        turbo = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
        turbo.enable_model_cpu_offload()   # a second SDXL-size model does not fit next to the first on a T4
        turbo.set_progress_bar_config(disable=True)
    return turbo(prompt, num_inference_steps=4, guidance_scale=0.0, generator=torch.Generator(DEVICE).manual_seed(42)).images[0], "sdxl-turbo (4 steps)"

p3_imgs, p3_rows = {}, []
for i, prompt in enumerate(P3_PROMPTS, 1):
    p3_imgs[("A", i)] = sdxl(prompt, scheduler="DPM++ 2M", steps=30, cfg=7.5, negative="blurry, low quality, distorted")
    sketch = layout_sketch(i - 1); sketch.save(f"images/part3/p{i}_layout.png")
    p3_imgs[("B", i)] = cn_generate(prompt, sketch, 0.7, steps=30, cfg=7.5)
    p3_imgs[("C", i)], c_name = config_c(prompt)
    for cfg in "ABC":
        p3_imgs[(cfg, i)] = p3_imgs[(cfg, i)].convert("RGB").resize((1024, 1024))
        p3_imgs[(cfg, i)].save(f"images/part3/p{i}_config{cfg}.png")
    print(f"prompt {i} done (C = {c_name})")
grid([p3_imgs[(c, i)] for i in range(1, 6) for c in "ABC"],
     [f"P{i} Config {c}" for i in range(1, 6) for c in "ABC"], "images/part3/all_configs_grid.png", cols=3, size=3.2)

In [ ]:
import lpips
lp = lpips.LPIPS(net="alex").to(DEVICE)
def to_t(img):
    a = np.asarray(img.resize((512, 512)), dtype=np.float32) / 127.5 - 1
    return torch.from_numpy(a).permute(2, 0, 1)[None].to(DEVICE)

def vision_rating(img, prompt):
    if not oa_client:
        return None, "no OpenAI key: rate manually"
    buf = io.BytesIO(); img.resize((512, 512)).save(buf, format="PNG")
    r = oa_client.chat.completions.create(model="gpt-4o", temperature=0, response_format={"type": "json_object"},
        messages=[{"role": "user", "content": [
            {"type": "text", "text": f"Prompt: {prompt}\nRate this image 1-5 for how well it follows the prompt (objects, counts, text, layout) and overall visual quality. Be strict: misspelled text or wrong counts cap the score at 3. Return JSON {{\"rating\": int, \"reason\": \"one sentence\"}}"},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}}]}])
    j = json.loads(r.choices[0].message.content); return j["rating"], j["reason"]

rows = []
for i, prompt in enumerate(P3_PROMPTS, 1):
    clips = {c: clip_score(p3_imgs[(c, i)], prompt) for c in "ABC"}
    ref_c = max(clips, key=clips.get)          # reference = highest-CLIP output for this prompt
    for c in "ABC":
        with torch.no_grad():
            d = lp(to_t(p3_imgs[(c, i)]), to_t(p3_imgs[(ref_c, i)])).item()
        rating, reason = vision_rating(p3_imgs[(c, i)], prompt)
        rows.append({"Config": c, "Prompt": f"P{i}: {prompt[:45]}", "CLIP Score": clips[c], "LPIPS vs best": round(d, 3),
                     "Reference": f"Config {ref_c}", "Rating 1-5": rating, "Notes": reason})
ev = pd.DataFrame(rows); save_table(ev, "part3_evaluation")
print(ev.groupby("Config")[["CLIP Score", "LPIPS vs best", "Rating 1-5"]].mean().round(2).to_markdown())
ev

In [ ]:
# README header: best 3 images (highest rating, then CLIP) across Part 3
top = ev.sort_values(["Rating 1-5", "CLIP Score"], ascending=False).head(3)
best = [p3_imgs[(r.Config, int(r.Prompt[1]))] for r in top.itertuples()]
grid(best, [f"{r.Config} | {r.Prompt[:30]}" for r in top.itertuples()], "images/best3_header.png", size=4)
import shutil
shutil.make_archive("week4_hw_images", "zip", ".", "images")
try:
    from google.colab import files; files.download("week4_hw_images.zip")
except Exception:
    print("zipped week4_hw_images.zip")

Written analyses: Part 1 and Part 2 memo in `README.md`, Part 3 report in `report.md`.